In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error


In [ ]:
#cargar el dataset
df = pd.read_csv("ventas_ml_clase2.csv")
df.head()

In [ ]:
print(f"filas: {df.shape[0]}, columnas: {df.shape[1]}")
print("")

print("Tipos de datos:")
for col, type in df.dtypes.items():
    print(f"{col:12s} {type}")
    


In [ ]:
df.describe(include="all").transpose().head()

In [ ]:
#predecir las ventas usando variables: marketing, precio, temporada, tienda, canal
x= df[["marketing", "precio", "temporada", "tienda", "canal"]]
y = df["ventas"]

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} filas, {X_train.shape[0]/len(df)*100:.0f}% del total")
print(f"Datos de prueba: {X_test.shape[0]} filas, {X_test.shape[0]/len(df)*100:.0f}% del total")


In [ ]:
numeric_features = ["marketing", "precio", "temporada"]
categorical_features = ["tienda", "canal"]

preprocess = ColumnTransformer(
    transformers=(
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    )
)

model = LinearRegression()

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

pipe

In [ ]:
pipe.fit(X_train, y_train) #entrenar el modelo

In [ ]:
#generar predicciones con los datos de prueba
y_pred = pipe.predict(X_test)  

mae = mean_absolute_error(y_test, y_pred) #error absoluto medio
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) #raiz del error cuadratico promedio
r2 = r2_score(y_test, y_pred) #R2: proporción de la varianza explicada por el modelo

media_ventas = y_test.mean() #promedio de ventas reales en el conjunto de prueba

print(f"MAE(Error absoluto medio): {mae:.2f}")
print(f"RMSE(Raíz del error cuadrático medio): {rmse:.2f}")
print(f"R2 (Proporción de varianza explicada): {r2:.4f}({r2*100:.2f}% )")

print("=" * 50)

print("Media de ventas reales:", f"${media_ventas:.2f}")
print("Error relativo (MAE/media):", f"{(mae/media_ventas)*100:.1f}%")

print("=" * 50)
print("interpretacion de las metricas:")
print(f"en promedio, el modelo se equivoca en ${mae:.2f} por prediccion.")
print(f"esto representa un {(mae/media_ventas)*100:.1f}% del promedio de ventas")
print(f"el modelo explica el {r2*100:.1f}% de la variabilidad en las ventas, lo cual es bastante bueno para un modelo simple")
print(f"el {(1-r2)*100:.1f}% restante se debe a factores no incluidos en el modelo")

if r2 >= 0.8:
    print("valoracion: buen ajuste para un modelo lineal")
elif r2 >= 0.5:
    print("valoracion: ajuste moderado, hay espacio para mejorar con modelos")
else:
    print("valoracion: ajuste bajo, se recomienda revisar las variables o probar otros modelos")



In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# --- Gráfica de Ventas Reales vs Predichas con línea de tendencia ---
fig, ax = plt.subplots(figsize=(8, 6))

# Cada punto es una observación del set de prueba
# Eje X = venta real, Eje Y = venta predicha por el modelo
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Predicciones")

# Línea de predicción perfecta (donde real = predicho)
# Si el modelo fuera perfecto, todos los puntos estarían sobre esta línea
limite_min = min(y_test.min(), y_pred.min()) - 10
limite_max = max(y_test.max(), y_pred.max()) + 10
ax.plot([limite_min, limite_max], [limite_min, limite_max],
        color="red", linestyle="--", linewidth=1.5,
        label="Prediccion perfecta (real = predicho)")

# Etiquetas y título
ax.set_xlabel("Ventas reales ($)", fontsize=12)
ax.set_ylabel("Ventas predichas ($)", fontsize=12)
ax.set_title(f"Ventas reales vs predichas  (R² = {r2:.2f})", fontsize=14)
ax.legend(fontsize=10)

# Ajustar los ejes para que tengan la misma escala
ax.set_xlim(limite_min, limite_max)
ax.set_ylim(limite_min, limite_max)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

# coefientes del modelo : que variables pesan mas?
un modelo de regresión lineal es una formula de **suma pondera**

ventas= base + (peso1 x marketing) + (peso 2 x precio) + (peso3 x temporada) +...

In [ ]:
# Extraer los coeficientes (peso) que el modelo aprendió

#obtener los nombres de las columnas categoricas despues del OneHotEncoding
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_cols = ohe.get_feature_names_out(["tienda", "canal"]).tolist()

#lista completa de variables: numericas +  las generadas por el OneHotEncoding
features_names = numeric_features + cat_cols

#los coeficientes son los "pesos" de la formula:
# ventas = intercepto + coefi1*marketing + coefi2*precio + ... + coefiN*canal_online
coef = pipe.named_steps["model"].coef_ #Coeficientes de cada variable

# organizar de mayor a menor
coef_df = pd.DataFrame({
    "feature": features_names,
    "coeficiente": coef
}).sort_values(by="coeficiente", key=abs, ascending=False)

#Mostrar la tabla de coeficientes

print("=" * 50)
print("Coeficientes del modelo:")
for _, fila in coef_df.iterrows():
          signo = "+" if fila["coeficiente"] >= 0 else "" 
          barra = "^" if fila["coeficiente"] >= 0 else "v"
          print(f" {barra} {fila['feature']:35s} {signo}{fila['coeficiente']:.4f}")
          